# VTuber Commercial Pipeline

이 노트북은 Google Colab에서 VTuber 아바타와 악세사리를 제작하기 위한 파이프라인입니다.

**두 가지 모드를 지원합니다:**

1. **Avatar Mode**: 캐릭터 이미지 → 3D VRM 아바타
2. **Accessory Mode**: 악세사리 이미지 → GLB 악세사리 + 장착 메타데이터

> 이미지 생성 AI는 이 노트북에 포함하지 않습니다. 입력 이미지는 사용자가 외부에서 생성해 가져옵니다.


---

## 1. Setup

환경 설정 및 필수 패키지 설치


In [ ]:
# 런타임 환경 확인
import os, subprocess, json, platform, sys

print('Python:', platform.python_version())
subprocess.run(['nvidia-smi'], check=False)
print('RAM/Disk:')
subprocess.run(['bash','-lc','free -h && df -h /content'], check=False)


In [ ]:
# 필수 패키지 설치
!pip -q install -U pip setuptools wheel
!pip -q install anime-face-detector pydantic pyyaml trimesh pillow click

# TripoSR 설치
%cd /content
!mkdir -p third_party
%cd /content/third_party
!test -d TripoSR || git clone https://github.com/VAST-AI-Research/TripoSR.git
%cd TripoSR
!pip -q install -r requirements.txt
!pip -q install -e .

# Virtual-pipeline 패키지 설치
subprocess.run([sys.executable, '-m', 'pip', 'install', 'git+https://github.com/jujumelona/Virtual-pipeline.git', '-q'], check=False)

print('설치 완료!')

In [ ]:
# 패키지 임포트 테스트
from vtuber_pipeline.core.config import BLOCKED_PACKAGES
from vtuber_pipeline.core.cache import PipelineCache
from vtuber_pipeline.avatar.face_detector import AnimeFaceDetector
from vtuber_pipeline.avatar.reconstruction import reconstruct_avatar
from vtuber_pipeline.accessory.attachment import AnchorType, generate_attachment_config
print('패키지 임포트 성공!')
print('차단된 패키지:', list(BLOCKED_PACKAGES.keys()))


In [ ]:
# 상업 프로필 안전장치
BLOCKED = {
    'instantmesh': 'runtime path uses nvdiffrast',
    'nvdiffrast': 'public license restricts non-NVIDIA use to non-commercial research/evaluation',
    'stable-fast-3d': 'community license has commercial conditions/revenue threshold',
}
print('상업적 사용 제한 패키지:')
print(json.dumps(BLOCKED, ensure_ascii=False, indent=2))


---

## 2. Avatar Mode

캐릭터 이미지를 업로드하고 VRM 아바타로 변환합니다.

**워크플로우:**
1. 이미지 업로드
2. 얼굴 검출 및 랜드마크 추출
3. 3D 메시 재구성 (TripoSR)
4. 템플릿 피팅 및 리깅
5. VRM 내보내기
6. 결과 미리보기 및 다운로드


### 2.1 이미지 업로드


In [ ]:
# 캐릭터 이미지 업로드
from google.colab import files
from pathlib import Path

uploaded = files.upload()
assert uploaded, '캐릭터 이미지를 업로드하세요.'
name = next(iter(uploaded))
Path('/content/input').mkdir(exist_ok=True)
avatar_image = Path('/content/input') / name
avatar_image.write_bytes(uploaded[name])
print('업로드된 아바타 이미지:', avatar_image)

### 2.2 얼굴 검출 및 랜드마크 추출


In [ ]:
# 얼굴 검출 및 랜드마크 추출
import cv2
from anime_face_detector import create_detector

img = cv2.imread(str(avatar_image))
assert img is not None, '이미지를 읽을 수 없습니다.'

detector = create_detector('yolov3')
preds = detector(img)
print('검출된 얼굴 수:', len(preds))

if len(preds) == 0:
    raise RuntimeError('애니 얼굴을 검출하지 못했습니다. 입력 프롬프트 조건에 맞춰 이미지를 다시 생성하세요.')

print('랜드마크 타입:', type(preds[0]))

### 2.3 3D 메시 재구성


In [ ]:
# TripoSR로 3D 재구성
from pathlib import Path
import subprocess

out_dir = Path('/content/workspace/avatar/reconstruct')
out_dir.mkdir(parents=True, exist_ok=True)

cmd = [
    'python', '/content/third_party/TripoSR/run.py',
    str(avatar_image),
    '--no-remove-bg',
    '--device', 'cuda:0',
    '--chunk-size', '8192',
    '--mc-resolution', '256',
    '--model-save-format', 'glb',
    '--output-dir', str(out_dir),
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)
print('3D 재구성 완료:', out_dir)

### 2.4 End-to-End VRM 파이프라인 실행


In [ ]:
# 전체 파이프라인 실행: 이미지 → VRM
from vtuber_pipeline.avatar.build import build_avatar
from google.colab import files

# build_avatar()로 전체 파이프라인 실행
result = build_avatar(
    image_path=str(avatar_image),
    output_dir='/content/output/',
    config={'profile': 'commercial', 'commercial_usage': 'corporation'}
)

print(f"Status: {result['status']}")
print(f"VRM: {result.get('vrm_path')}")

# 다운로드
if result['status'] == 'complete' and result.get('vrm_path'):
    files.download(result['vrm_path'])
else:
    print('Pipeline did not complete successfully')
    print('Stage results:')
    for stage, stage_result in result.get('stages', {}).items():
        print(f"  {stage}: {stage_result.get('status', 'unknown')}")

### 2.5 아바타 3D 재구성 실행 (개별 단계)


In [ ]:
# 아바타 3D 재구성 실행 (패키지 API)
from vtuber_pipeline.avatar.reconstruction import reconstruct_avatar
from vtuber_pipeline.avatar.face_detector import AnimeFaceDetector

# 얼굴 검출 및 랜드마크 추출
detector = AnimeFaceDetector()
landmarks = detector.detect(str(avatar_image))
print(f'검출된 얼굴 수: {len(landmarks)}')

# 3D 재구성
result = reconstruct_avatar(
    image_path=str(avatar_image),
    output_dir='/content/workspace/avatar',
    profile='commercial'
)
print('재구성 완료:', result)

### 2.5 결과 미리보기


In [ ]:
# 결과 파일 확인
from pathlib import Path
import glob

workspace = Path('/content/workspace/avatar')
files = list(workspace.rglob('*'))
print('생성된 파일:')
for f in files:
    if f.is_file():
        print(f'  {f.relative_to(workspace)} ({f.stat().st_size} bytes)')

### 2.6 다운로드


In [ ]:
# 결과 다운로드
from google.colab import files as colab_files
import shutil

# ZIP으로 패키징
output_zip = '/content/avatar_output.zip'
shutil.make_archive('/content/avatar_output', 'zip', '/content/workspace/avatar')
colab_files.download(output_zip)

---

## 3. Accessory Mode

악세사리 이미지를 업로드하고 GLB 파일로 변환합니다.

**워크플로우:**
1. 베이스 VRM 업로드 (선택)
2. 악세사리 이미지 업로드
3. 3D 재구성
4. 장착 앵커 선택
5. attachment.json 생성
6. 결과 다운로드


### 3.1 베이스 VRM 업로드 (선택)


In [ ]:
# 베이스 VRM 업로드 (선택사항)
from google.colab import files
from pathlib import Path

print('베이스 VRM을 업로드하세요 (건너뛰려면 취소 클릭)')
base_vrm_upload = files.upload()

if base_vrm_upload:
    base_vrm_name = next(iter(base_vrm_upload))
    Path('/content/input/base').mkdir(parents=True, exist_ok=True)
    base_vrm = Path('/content/input/base') / base_vrm_name
    base_vrm.write_bytes(base_vrm_upload[base_vrm_name])
    print('업로드된 베이스 VRM:', base_vrm)
else:
    base_vrm = None
    print('베이스 VRM 없이 진행')

### 3.2 악세사리 이미지 업로드


In [ ]:
# 악세사리 이미지 업로드 (여러 개 가능)
from google.colab import files
from pathlib import Path

print('악세사리 이미지를 업로드하세요 (여러 개 선택 가능)')
uploaded_acc = files.upload()

acc_dir = Path('/content/input/accessories')
acc_dir.mkdir(parents=True, exist_ok=True)
accessory_images = []
for name, data in uploaded_acc.items():
    p = acc_dir / name
    p.write_bytes(data)
    accessory_images.append(p)
print(f'업로드된 악세사리 {len(accessory_images)}개:', accessory_images)

### 3.3 악세사리 3D 재구성


In [ ]:
# 악세사리 일괄 3D 재구성
from vtuber_pipeline.accessory.batch import batch_reconstruct_accessories

if accessory_images:
    results = batch_reconstruct_accessories(
        image_paths=[str(p) for p in accessory_images],
        output_dir='/content/workspace/accessories',
        profile='commercial'
    )
    print('재구성된 악세사리:', results)

### 3.4 장착 앵커 선택


In [ ]:
# 지원 앵커 타입 안내
from vtuber_pipeline.accessory.attachment import AnchorType

print('지원 앵커 타입:')
for anchor in AnchorType:
    print(f'  - {anchor.value}')

print('''
앵커 타입 설명:
- HEAD_TOP: 왕관, 모자, 고양이귀
- FACE: 안경, 마스크
- LEFT_EAR, RIGHT_EAR: 귀걸이
- NECK: 목걸이
- CHEST: 브로치
- BACK: 날개, 가방
- LEFT_HAND, RIGHT_HAND: 손에 쥐는 아이템
- HIPS: 꼬리
''')

### 3.5 장착 설정 생성


In [ ]:
# 장착 설정 생성
from vtuber_pipeline.accessory.attachment import generate_attachment_config

# 앵커 선택 (예: HEAD_TOP)
selected_anchor = AnchorType.HEAD_TOP

attachment_config = generate_attachment_config(
    name='accessory',
    anchor=selected_anchor,
    translation=[0.0, 0.13, 0.01],
    rotation=[0.0, 0.0, 0.0],
    scale=[0.11, 0.11, 0.11]
)
print('장착 설정:', attachment_config)

# attachment.json 저장
attachment_path = Path('/content/workspace/accessories/attachment.json')
attachment_path.write_text(json.dumps(attachment_config, indent=2))
print('저장됨:', attachment_path)

### 3.6 결과 다운로드


In [ ]:
# 악세사리 결과 다운로드
from google.colab import files as colab_files
import shutil

# ZIP으로 패키징
output_zip = '/content/accessory_output.zip'
shutil.make_archive('/content/accessory_output', 'zip', '/content/workspace/accessories')
colab_files.download(output_zip)

---

## 4. 권장 프롬프트

외부 이미지 생성 AI에서 사용할 권장 프롬프트입니다.


### Avatar 프롬프트

```text
[CHARACTER DESCRIPTION]

Create a clean VTuber character reference image for 3D reconstruction.
Single character only. Front-facing or near-front-facing camera.
Head, hair silhouette, neck, shoulders and upper torso fully visible.
Waist-up composition preferred. Neutral expression.
Eyes and mouth clearly visible. Arms not blocking torso.
No large prop covering the body. Simple uniform background.
Even lighting. Minimal perspective distortion. High facial detail.
```

### Accessory 프롬프트

```text
[ACCESSORY DESCRIPTION]

Create one isolated wearable or handheld accessory for 3D reconstruction.
Single object only. Fully visible. Centered. Plain uniform background.
No person, no mannequin, no hand holding it, no occlusion.
Neutral canonical orientation. Even lighting. Minimal perspective distortion.
```


---

## 5. 검증 체크리스트

완성된 아바타 검증 항목입니다.


In [ ]:
# 아바타 검증 체크리스트
required_checks = [
    ('VRM parse', 'VRM 파일이 정상적으로 파싱됨'),
    ('required humanoid bones', '필수 humanoid bone이 모두 존재'),
    ('head/neck rotation', '머리/목 회전이 정상'),
    ('blink / blinkLeft / blinkRight', '블링크 표정이 설정됨'),
    ('aa / ih / ou / ee / oh', '리싱크 표정이 설정됨'),
    ('happy / angry / sad / relaxed / surprised', '감정 표정이 설정됨'),
    ('eye gaze', '시선 추적이 설정됨'),
    ('springbone references', '스프링본이 설정됨'),
    ('textures/materials', '텍스처/머티리얼이 포함됨'),
    ('preview renders', '미리보기 렌더가 생성됨'),
]

print('아바타 검증 체크리스트:')
for name, desc in required_checks:
    print(f'  [ ] {name}: {desc}')